# Laya-CUA: fine-tune Laya for computer-use decisions (free GPU, single T4)

Fine-tunes [`convaiinnovations/laya`](https://huggingface.co/convaiinnovations/laya) (Apache-2.0, 421M-param ModernBERT-large encoder plus a typed decision head) to pick the **next computer-use action** from a screen state.

**What this is, and is not.** Laya is a non-autoregressive decision model. It never generates text, so it cannot emit click coordinates or typed strings. It scores a fixed set of typed options in one forward pass. Here that means: given the task, the screen (accessibility tree or screen text) and recent history, choose the next action type, whether the task is finished, and whether to escalate to a human. A generative agent (a VLM or LLM) still produces the arguments. Laya is the fast decision layer in front of it.

**Training method.** Uses the official `laya.train` primitives (RLCD, proper-scoring-rule rewards, temperature calibration), same as the upstream typed-decisions notebook. Unsloth does not apply: it targets autoregressive LLMs, not this encoder.

**Data.** You provide it. No data ships with this notebook. Format below. Nothing here reports results until you run it.

Runs on a free Kaggle T4 or Colab T4. Cost: $0.

## 1. Environment

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Switch the runtime to a GPU (T4)."
print(torch.cuda.get_device_name(0), round(torch.cuda.get_device_properties(0).total_memory/1e9,1), "GB")

In [ ]:
!pip install -q -U "laya>=0.1.6" "transformers>=4.48.0" "datasets>=3.0.0" safetensors huggingface_hub pandas scipy accelerate
import laya; print("laya", laya.__version__)

## AgentNet converter
[OpenCUA AgentNet](https://huggingface.co/datasets/xlangai/AgentNet) (MIT) has per-step screen descriptions, action text and `pyautogui` code. The converter below turns each step into one row: the state is the task, the text description of the current screen and the last few actions; the label is the action type parsed from the step's code (`click`/`type`/`key`/`scroll`/`move`/`drag`/`wait`/`done`), and `task_done` is true on a successful `terminate`. The step's own `thought` and `action` text are left out of the state because they would give the answer away. Screenshots are not used. `needs_human` has no label in AgentNet and is skipped. Steps marked incorrect by the annotators are dropped.

In [ ]:
%%writefile agentnet_to_laya.py
"""Convert OpenCUA AgentNet trajectories (MIT) into Laya typed-decision rows.

Each trajectory step becomes one row:
  state = {task, screen (text description of the current screen), history (previous action texts)}
  expected = {action_type: label from the step's pyautogui code, task_done: true on a successful terminate}
The step's own `thought` and `action` text are NOT in the state (they describe the answer).
`needs_human` has no label in AgentNet, so it is left unlabelled and skipped in training.
Screenshots are not used: Laya reads text only.
"""
import json, re, random, itertools

ACTION_MAP = {
    "click": "click", "doubleClick": "click", "tripleClick": "click", "rightClick": "click", "middleClick": "click",
    "moveTo": "move", "write": "type", "typewrite": "type", "press": "key", "hotkey": "key", "keyDown": "key", "keyUp": "key",
    "scroll": "scroll", "hscroll": "scroll", "dragTo": "drag", "drag": "drag",
    "terminate": "done", "wait": "wait", "sleep": "wait",
}

def parse_code(code):
    """Return (action_type, success_flag_or_None), or (None, None) if unknown."""
    code = (code or "").strip()
    m = re.match(r"(?:pyautogui|computer)\.(\w+)", code)
    if not m:
        return None, None
    name = m.group(1)
    kind = ACTION_MAP.get(name)
    if kind is None:
        return None, None
    ok = None
    if name == "terminate":
        ok = "success" in code
    return kind, ok

def trajectory_rows(task, history_len=4, max_obs_chars=1800, require_correct=True):
    traj = task.get("traj") or []
    instr = task.get("instruction") or task.get("natural_language_task") or ""
    rows, hist = [], []
    for step in traj:
        v = step.get("value") or {}
        kind, ok = parse_code(v.get("code"))
        text = (v.get("action") or "").strip()
        if kind is None or not v.get("observation"):
            if text: hist.append(text[:160])
            continue
        if not (require_correct and v.get("last_step_correct") is False):
            rows.append({
                "task_id": task.get("task_id"),
                "state": {"task": instr, "screen": v["observation"][:max_obs_chars], "history": hist[-history_len:]},
                "expected": {"action_type": kind, "task_done": bool(ok)},
            })
        if text: hist.append(text[:160])
    return rows

def stream_agentnet(n_tasks=600, file="agentnet_ubuntu_5k.jsonl", repo="xlangai/AgentNet", seed=0, skip=0):
    """Stream the first lines of an AgentNet jsonl over HTTP and stop after n_tasks (no 200 GB download)."""
    import requests
    url = f"https://huggingface.co/datasets/{repo}/resolve/main/{file}"
    out, kept = [], 0
    with requests.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        for i, line in enumerate(r.iter_lines()):
            if i < skip or not line: continue
            try: task = json.loads(line)
            except Exception: continue
            rows = trajectory_rows(task)
            if rows: out.extend(rows); kept += 1
            if kept >= n_tasks: break
    return out

def split_by_task(rows, eval_frac=0.1, seed=7):
    """Hold out whole tasks, so no task leaks across train and eval."""
    ids = sorted({r["task_id"] for r in rows}); random.Random(seed).shuffle(ids)
    ev = set(ids[:max(1, int(len(ids) * eval_frac))])
    return [r for r in rows if r["task_id"] not in ev], [r for r in rows if r["task_id"] in ev]

## 2. Dataset format

One JSON object per line (`.jsonl`). Each row is one decision point from a recorded computer-use trajectory:

```json
{"state": {"task": "...", "screen": "<accessibility tree or visible text>", "history": ["step 1 ...", "step 2 ..."]},
 "questions": QUESTIONS,
 "expected": {"action_type": "click", "task_done": false, "needs_human": false}}
```

`QUESTIONS` is the fixed schema defined in the next cell. The same schema is used for every row, so you only provide `state` and `expected`. Set `DATA_PATH` to your file, or set `HF_DATASET` to a Hub dataset with `state` and `expected` columns (JSON strings or objects).

If your data is a plain `text,label` CSV for a single question, use `CSV_PATH` instead.

In [ ]:
import json, os

# ---- EDIT THESE ----
USE_AGENTNET = True                                  # OpenCUA AgentNet sample (MIT), streamed, no 200 GB download
N_TASKS      = 400                                   # tasks to stream (~1 step = 1 row); raise if time allows
AGENTNET_FILE = "agentnet_ubuntu_5k.jsonl"           # or "agentnet_win_mac_18k.jsonl"
DATA_PATH   = None                                   # or your own jsonl ({state, expected})
EVAL_PATH   = None                                   # optional separate held-out jsonl; else 10% is split off
HF_DATASET  = None                                   # e.g. "sodan/cua-trajectories" (needs state, expected cols)
HF_REPO     = "sodan/laya-cua"                    # target model repo on the Hub
EPOCHS      = 2
MAX_STEPS = None   # cap total optimizer steps for a quick run; None = full EPOCHS
# --------------------

ACTIONS = {
  "click":      "click or tap a UI element",
  "type":       "type text into a focused field",
  "key":        "press a key or shortcut",
  "scroll":     "scroll the view",
  "navigate":   "open a URL or switch app or tab",
  "move":       "move the mouse without clicking (hover)",
  "wait":       "wait for the page or app to load",
  "drag":       "drag and drop or select by dragging",
  "done":       "the task is already complete, stop",
}
QUESTIONS = {
  "action_type": {"type": "choice", "instructions": "Which action type should the agent take next?", "criteria": ACTIONS},
  "task_done":   {"type": "noul",   "instructions": "Is the task already fully complete on screen?"},
  "needs_human": {"type": "noul",   "instructions": "Should a human review before continuing (payment, login, irreversible or unclear step)?"},
}

In [ ]:
import os, json, random, torch
from transformers import AutoTokenizer
from huggingface_hub import snapshot_download
from laya.agent import _fix_tokenizer_config
from laya.train import items_from_rows

def load_rows(path=None, hf=None):
    rows = []
    if path:
        with open(path) as f:
            for line in f:
                if not line.strip(): continue
                r = json.loads(line)
                rows.append({"state": r["state"], "questions": QUESTIONS, "expected": r["expected"]})
    if hf:
        from datasets import load_dataset
        for r in load_dataset(hf, split="train"):
            j = lambda v: json.loads(v) if isinstance(v, str) else v
            rows.append({"state": j(r["state"]), "questions": QUESTIONS, "expected": j(r["expected"])})
    return rows

from agentnet_to_laya import stream_agentnet, split_by_task
rows = []
if USE_AGENTNET:
    an = stream_agentnet(N_TASKS, AGENTNET_FILE)
    rows += [{"task_id": r["task_id"], "state": r["state"], "questions": QUESTIONS, "expected": r["expected"]} for r in an]
rows += load_rows(DATA_PATH if DATA_PATH and os.path.exists(DATA_PATH) else None, HF_DATASET)
assert rows, "No data loaded."
if EVAL_PATH:
    eval_rows = load_rows(EVAL_PATH); train_rows = rows
else:
    train_rows, eval_rows = split_by_task(rows)   # whole tasks held out, so steps of one task never sit in both
import collections
print("train action mix:", collections.Counter(r["expected"]["action_type"] for r in train_rows).most_common())
print(len(train_rows), "train rows,", len(eval_rows), "held-out rows")

MODEL_ID = "convaiinnovations/laya"
model_dir = snapshot_download(MODEL_ID)
_fix_tokenizer_config(model_dir)
tok = AutoTokenizer.from_pretrained(os.path.join(model_dir, "tokenizer"))
cfg = json.load(open(os.path.join(model_dir, "rl_agent_config.json")))
items, skipped = items_from_rows(tok, train_rows, cfg["max_len"], cfg["head_max_len"])
print(len(items), "train items; skipped:", skipped)
assert items, "No usable items: check state/expected format."

## 3. Train (RLCD, single GPU, fp16 autocast) and calibrate

In [ ]:
import time, math
from safetensors.torch import load_file
from laya.common import build_model, collate_items
from laya.calibrate import fit_temperature_map
from laya.train import calibration_records, encode_item, rlcd_loss, save_checkpoint, sigma_at, split_calibration

OUTPUT_DIR = "/kaggle/working/laya_cua" if os.path.exists("/kaggle") else "/content/laya_cua"
device = torch.device("cuda")
cfg = dict(cfg, gradient_checkpointing=True, max_tokens_per_batch=4096, max_len=1024, head_max_len=256)
max_len, head_max_len = cfg["max_len"], cfg["head_max_len"]

model = build_model(cfg, encoder_dir=os.path.join(model_dir, "encoder"))
model.load_state_dict(load_file(os.path.join(model_dir, "model.safetensors")), strict=True)
model.encoder.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
model.head_checkpointing = True
model.to(device).train()

train_items, calib_items = split_calibration(items, calib_max=400, calib_frac=0.1, seed=20261118)
MICRO_BATCH, GRAD_ACCUM, GROUP_SIZE = 8, 8, 4      # effective batch 64
LR_ENC, LR_HEAD, SIGMA0, SIGMA1 = 2.5e-5, 1e-4, 0.4, 0.1
enc = [p for n,p in model.named_parameters() if "encoder." in n]
head = [p for n,p in model.named_parameters() if "encoder." not in n]
opt = torch.optim.AdamW([{"params":enc,"lr":LR_ENC},{"params":head,"lr":LR_HEAD}], weight_decay=0.01)
total = max(1, (len(train_items)//(MICRO_BATCH*GRAD_ACCUM))*EPOCHS)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=total, eta_min=1e-6)
BF16_SUPPORTED = torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16_SUPPORTED else torch.float16
use_scaler = not BF16_SUPPORTED
scaler = torch.amp.GradScaler("cuda") if use_scaler else None
print(f"AMP dtype: {AMP_DTYPE}; GradScaler {'enabled' if use_scaler else 'disabled'}")

torch.cuda.reset_peak_memory_stats()
start_reserved = torch.cuda.memory_reserved()
t0 = time.time()
optimizer_steps = 0
for epoch in range(EPOCHS):
    random.Random(42+epoch).shuffle(train_items)
    sigma = sigma_at(epoch, EPOCHS, SIGMA0, SIGMA1)
    opt.zero_grad(set_to_none=True); step = 0; tot = 0.0
    nb = math.ceil(len(train_items)/MICRO_BATCH)
    for b in range(0, len(train_items), MICRO_BATCH):
        chunk = train_items[b:b+MICRO_BATCH]
        batch = collate_items([[encode_item(tok, it, max_len, head_max_len) for it in chunk]], tok.pad_token_id)
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits, act = model(batch["input_ids"].to(device), batch["attention_mask"].to(device),
                                batch["marker_pos"].to(device), batch["marker_mask"].to(device), batch["qtype"].to(device))
        loss = rlcd_loss(logits.float(), batch["target"].to(device), batch["marker_mask"].to(device),
                         batch["qtype"].to(device), sigma, samples=GROUP_SIZE, w_sph=0.75, w_rps=1.0)
        win = min(GRAD_ACCUM, nb - (step//GRAD_ACCUM)*GRAD_ACCUM)
        if use_scaler:
            scaler.scale(loss/win).backward()
        else:
            (loss/win).backward()
        step += 1; tot += loss.item()
        if step % GRAD_ACCUM == 0 or b+MICRO_BATCH >= len(train_items):
            if use_scaler:
                scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            if use_scaler:
                scaler.step(opt)
                scaler.update()
            else:
                opt.step()
            sched.step(); opt.zero_grad(set_to_none=True)
            optimizer_steps += 1
            if MAX_STEPS is not None and optimizer_steps >= MAX_STEPS:
                break
    print(f"epoch {epoch+1}/{EPOCHS} avg loss {tot/max(1,step):.4f} ({time.time()-t0:.0f}s)")
    save_checkpoint(model, tok, cfg, os.path.join(OUTPUT_DIR, "checkpoint_latest"))
    if MAX_STEPS is not None and optimizer_steps >= MAX_STEPS:
        break

model.eval(); torch.cuda.empty_cache()
fitted = fit_temperature_map(calibration_records(model, tok, calib_items, device, max_len, head_max_len, batch_size=16))
out_cfg = dict(cfg, fine_tuned=True, model_name="laya-cua", temperature=fitted["temperature"])
out_cfg.pop("temperature_by_options", None)
if fitted["temperature_by_options"]: out_cfg["temperature_by_options"] = fitted["temperature_by_options"]
save_checkpoint(model, tok, out_cfg, OUTPUT_DIR)
print("saved", OUTPUT_DIR, "temps", fitted["temperature"])
peak_reserved_gb = torch.cuda.max_memory_reserved() / 1e9
total_gpu_gb = torch.cuda.get_device_properties(device).total_memory / 1e9
peak_reserved_pct = 100 * peak_reserved_gb / total_gpu_gb
print(f"peak reserved: {peak_reserved_gb:.2f} GB")
print(f"total GPU: {total_gpu_gb:.2f} GB")
print(f"peak reserved: {peak_reserved_pct:.1f}% of total")
print(f"total train wall-clock: {time.time() - t0:.1f}s")

## 4. Evaluate on held-out rows

Accuracy per question on rows the model never trained on, with the untuned base model as a reference. Report these numbers as measured, with the dataset size. Calibration temperatures were fit on a slice of the training items, so this held-out set is the one to quote.

In [ ]:
import numpy as np, laya
def evaluate(agent, rows):
    ok = {k: [] for k in QUESTIONS}; confs=[]
    for r in rows:
        res = agent.predict(r["state"], QUESTIONS)
        for qid, g in r["expected"].items():
            a = res["answers"][qid]
            if QUESTIONS[qid]["type"] == "choice":
                ok[qid].append(a["choice"] == str(g))
            else:
                ok[qid].append((a["noul"] >= 0.5) == bool(g))
    return {k: (round(float(np.mean(v)),4), len(v)) for k, v in ok.items() if v}

ft   = laya.Agent(OUTPUT_DIR, device="cuda")
base = laya.Agent(MODEL_ID, device="cuda")
maj = collections.Counter(r["expected"]["action_type"] for r in train_rows).most_common(1)[0]
print("majority-class baseline for action_type:", maj[0], round(sum(r["expected"]["action_type"]==maj[0] for r in eval_rows)/len(eval_rows),4))
print("fine-tuned:", evaluate(ft, eval_rows))
print("base      :", evaluate(base, eval_rows))

## 5. Save to Google Drive (default)
Copies the trained model to `MyDrive/laya_cua` in the signed-in Google account, so nothing depends on Hugging Face. Download it from Drive or push it to the Hub later (section 6).

In [ ]:
import shutil, os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    dst = "/content/drive/MyDrive/laya_cua"
except Exception:
    dst = "/kaggle/working/laya_cua_final"   # Kaggle: the working dir is saved with the notebook output
if os.path.abspath(dst) != os.path.abspath(OUTPUT_DIR):
    shutil.copytree(OUTPUT_DIR, dst, dirs_exist_ok=True, ignore=shutil.ignore_patterns("checkpoint_latest"))
print("saved to", dst, os.listdir(dst))

## 6. Optional: push to Hugging Face (run only when you want to)
Not needed for the model to exist; section 5 already saved it. Needs a **Write** token for the account that owns `HF_REPO`: add `HF_TOKEN` in Colab Secrets, or paste it into the login box that appears.

In [ ]:
import os
from huggingface_hub import HfApi
token = None
try:
    from kaggle_secrets import UserSecretsClient; token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:
        from google.colab import userdata; token = userdata.get("HF_TOKEN")
    except Exception:
        token = os.environ.get("HF_TOKEN")
if not token:
    from huggingface_hub import notebook_login
    notebook_login()          # opens a login box; paste a Write token there yourself
    from huggingface_hub import get_token; token = get_token()
assert token, "No Hugging Face token."
api = HfApi(token=token)
api.create_repo(HF_REPO, repo_type="model", exist_ok=True)

card = f'''---
license: apache-2.0
base_model: convaiinnovations/laya
library_name: laya
tags: [laya, computer-use, decision-model, calibrated]
---
# laya-cua

`convaiinnovations/laya` fine-tuned to choose the next computer-use action type from a screen state. It is a typed decision model: it scores {list(ACTIONS)} plus `task_done` and `needs_human`. It does not generate text, coordinates or typed strings.

Training: RLCD via `laya.train`, {len(train_items)} items, {EPOCHS} epochs. Held-out evaluation: see the notebook output; numbers are only valid for the dataset it was run on.

Notebook and code: https://github.com/somdipto/laya-cua
'''
open(os.path.join(OUTPUT_DIR, "README.md"), "w").write(card)
api.upload_folder(folder_path=OUTPUT_DIR, repo_id=HF_REPO, repo_type="model", ignore_patterns=["checkpoint_latest/*"])
print("https://huggingface.co/" + HF_REPO)